# exp_wgt_return_6m 防御性+收益补偿策略

In [6]:
# =====================================================
# exp_wgt_return_6m 策略：15 市值组 + 指定组别内因子低分位等权
# 提交模拟版：动态结束日期 + EFFECTIVE_END_DATE + 深度优化信号结构
#
# 用途：
#   放入 BigQuant AIStudio 的单独 .ipynb 中运行，回测成功后提交日频模拟。
#
# 核心设计：
#   1）END_DATE 每次运行自动取当天日期；提交模拟后，平台未来每天运行时会自动扩展到新的可用交易日。
#   2）BigTrader 的 end_date 使用 EFFECTIVE_END_DATE，即信号表中实际存在的最后交易日，避免非交易日、未来日、数据未落库导致失败。
#   3）保持策略逻辑参数不变：市值15组、SELECT_MCAP_GROUPS=[1]、每组低分位10%、20日调仓、趋势仓位、行业过滤等均不改变。
#   4）T 日收盘后形成选股与趋势信号，T+1 日开盘执行；不使用 m_lead / next_open 等未来字段。
#   5）只在调仓信号日计算最终入选股票，不生成全市场每日候选大表。
#   6）只向 BigTrader 传入交易必要字段，并在 initialize 中预索引为 date -> target_map，避免 handle_data 每日扫描 DataFrame。
# =====================================================

import math
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

import dai
from bigquant import bigtrader


# =====================================================
# 1. 策略参数区
# =====================================================

# 回测起点只用于构造历史信号和固定调仓节奏；不要在提交模拟后频繁改动，否则会改变调仓相位。
START_DATE = "2023-01-01"

# 提交模拟必须动态取当前日期：未来每次模拟任务运行时，都会自动向后扩展查询区间。
# 若当天是非交易日，或当天行情尚未落库，后面会自动回落到 EFFECTIVE_END_DATE。
END_DATE = datetime.today().strftime("%Y-%m-%d")

# 市值分组数量：按 total_market_cap 从小到大分成 15 组
# 组号含义：1 = 最小市值组，15 = 最大市值组
MCAP_GROUP_COUNT = 15

# 指定参与选股的市值组，例如 [1, 2, 3] 表示只在最小的 3 个市值组中选股
SELECT_MCAP_GROUPS = [1]

# 每个指定市值组内，选择因子值最低的比例
GROUP_SELECT_PCT = 0.10

# 每隔多少个交易日重新选股一次
REBALANCE_DAYS = 20

# 强势市场目标总仓位
TARGET_TOTAL_WEIGHT = 0.98

# 弱势市场防御总仓位；如果想弱势时完全空仓，改成 0.0
DEFENSIVE_TOTAL_WEIGHT = 0.30

# 趋势过滤指数；小市值策略通常可使用中证1000
TREND_INDEX = "000852.SH"

# 指数趋势均线天数
TREND_MA_DAYS = 30

# exp_wgt_return_6m：6个月成交量加权衰减动量因子
FACTOR_MONTHS = 6
LOOKBACK_DAYS = 21 * FACTOR_MONTHS

# 为计算 m_lag 和指数均线，需要在回测开始日前多取一段历史
BEFORE_START_DAYS = max(300, TREND_MA_DAYS * 3)

# list_days 是自然日口径，不是交易日口径
MIN_LIST_DAYS = int(LOOKBACK_DAYS * 365 / 252) + 30

# 初始资金
CAPITAL_BASE = 1_000_000

# 剔除行业：使用中信一级行业 cs_level1_name
EXCLUDE_INDUSTRIES = [
    "交通运输",
    "电力及公用事业",
    "纺织服装",
    "轻工制造",
    "家电",
    "石油石化",
    "综合",
    "银行",
]

# 回测基准
BENCHMARK = "000300.SH"

# 是否打印每日调仓 / 风控日志
VERBOSE = False

# 手续费参数，需要和 context.set_commission 保持一致
BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COST = 5.0

# 使用当前 Bar 撮合：T 日收盘后形成信号，T+1 日开盘读取当前 Bar 判断涨跌停并成交
USE_CURRENT_BAR_MATCHING = True

# 浮点比较容忍度
EPS = 1e-10


# =====================================================
# 2. 通用辅助函数
# =====================================================

def _normalize_mcap_groups(groups, group_count):
    if groups is None or len(groups) == 0:
        raise ValueError("SELECT_MCAP_GROUPS 不能为空，例如 [1, 2, 3]。")

    normalized = sorted({int(g) for g in groups})
    invalid = [g for g in normalized if g < 1 or g > group_count]
    if invalid:
        raise ValueError(
            f"SELECT_MCAP_GROUPS 中存在非法组别 {invalid}；"
            f"有效范围为 1 到 {group_count}。"
        )
    return normalized


def _sql_quote(values):
    if values is None or len(values) == 0:
        raise ValueError("SQL 列表不能为空。")
    return ", ".join(["'" + str(x).replace("'", "''") + "'" for x in values])


def _sql_int_list(values):
    if values is None or len(values) == 0:
        raise ValueError("SQL 整数列表不能为空。")
    return ", ".join([str(int(x)) for x in values])


def lag(field: str, k: int) -> str:
    if k == 0:
        return field
    return f"m_lag({field}, {k})"


def _safe_float(x, default=np.nan):
    try:
        if x is None or pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def _validate_parameters():
    global SELECT_MCAP_GROUPS
    SELECT_MCAP_GROUPS = _normalize_mcap_groups(SELECT_MCAP_GROUPS, MCAP_GROUP_COUNT)

    if not (0 < GROUP_SELECT_PCT <= 1):
        raise ValueError("GROUP_SELECT_PCT 必须位于 (0, 1] 区间内。")
    if REBALANCE_DAYS <= 0:
        raise ValueError("REBALANCE_DAYS 必须为正整数。")


def _build_exp_wgt_return_expr():
    num_terms = []
    den_terms = []

    for i in range(LOOKBACK_DAYS):
        decay_weight = math.exp(-i / FACTOR_MONTHS / 4.0)
        close_i = lag("close", i)
        close_i_1 = lag("close", i + 1)
        turn_i = lag("turn", i)
        ret_i = f"(({close_i} / NULLIF({close_i_1}, 0)) - 1.0)"

        num_terms.append(f"COALESCE(({decay_weight:.12g} * {turn_i} * {ret_i}), 0.0)")
        den_terms.append(f"COALESCE(({decay_weight:.12g} * {turn_i}), 0.0)")

    return " + ".join(num_terms), " + ".join(den_terms)


# =====================================================
# 3. 信号构造：动态 END_DATE + 实际 EFFECTIVE_END_DATE
# =====================================================

def build_signal_data():
    _validate_parameters()

    calc_start_date = (
        datetime.strptime(START_DATE, "%Y-%m-%d") - timedelta(days=BEFORE_START_DAYS)
    ).strftime("%Y-%m-%d")

    industry_sql = _sql_quote(EXCLUDE_INDUSTRIES)
    selected_mcap_group_sql = _sql_int_list(SELECT_MCAP_GROUPS)
    num_expr, den_expr = _build_exp_wgt_return_expr()

    # 趋势信号在 date 收盘后才可得，因此交易时使用上一交易日的趋势结果。
    index_trend_sql = f"""
    WITH index_raw AS (
        SELECT
            date,
            close AS trend_index_close,
            AVG(close) OVER (
                PARTITION BY instrument
                ORDER BY date
                ROWS BETWEEN {TREND_MA_DAYS - 1} PRECEDING AND CURRENT ROW
            ) AS trend_index_ma
        FROM cn_stock_index_bar1d
        WHERE instrument = '{TREND_INDEX}'
          AND date >= '{calc_start_date}'
          AND date <= '{END_DATE}'
    ),
    index_trend AS (
        SELECT
            date,
            trend_index_close,
            trend_index_ma,
            CASE
                WHEN trend_index_ma IS NULL THEN 0
                WHEN trend_index_close > trend_index_ma THEN 1
                ELSE 0
            END AS risk_on,
            CASE
                WHEN trend_index_ma IS NULL THEN {DEFENSIVE_TOTAL_WEIGHT}
                WHEN trend_index_close > trend_index_ma THEN {TARGET_TOTAL_WEIGHT}
                ELSE {DEFENSIVE_TOTAL_WEIGHT}
            END AS trend_total_weight
        FROM index_raw
    )
    SELECT
        date,
        trend_index_close,
        trend_index_ma,
        risk_on,
        trend_total_weight
    FROM index_trend
    WHERE date >= '{START_DATE}'
    ORDER BY date ASC
    """

    daily_trend_raw = dai.query(index_trend_sql).df()
    daily_trend_raw["date"] = pd.to_datetime(daily_trend_raw["date"]).dt.strftime("%Y-%m-%d")
    daily_trend_raw = daily_trend_raw.sort_values("date").reset_index(drop=True)

    if daily_trend_raw.empty:
        raise ValueError("daily_trend_raw 为空，请检查趋势指数代码或日期范围。")

    all_dates = daily_trend_raw["date"].tolist()
    if len(all_dates) < 2:
        raise ValueError("交易日数量不足，无法形成 T 日信号、T+1 日交易。")

    # T 日信号移动到 T+1 交易日执行。
    daily_trend_trade = daily_trend_raw.copy()
    daily_trend_trade["trend_signal_date"] = daily_trend_trade["date"].shift(1)
    for col in ["trend_index_close", "trend_index_ma", "risk_on", "trend_total_weight"]:
        daily_trend_trade[col] = daily_trend_trade[col].shift(1)

    daily_trend_trade = daily_trend_trade.dropna(
        subset=["trend_signal_date", "trend_total_weight"]
    ).copy()
    daily_trend_trade["risk_on"] = daily_trend_trade["risk_on"].astype(np.int8)
    daily_trend_trade["trend_total_weight"] = daily_trend_trade["trend_total_weight"].astype(np.float32)

    # 调仓信号在 all_dates 中按固定相位生成。START_DATE 固定后，未来新增日期只会自然延长调仓序列。
    rebalance_signal_dates = all_dates[::REBALANCE_DAYS]
    next_date_map = {all_dates[i]: all_dates[i + 1] for i in range(len(all_dates) - 1)}
    rebalance_signal_dates = [d for d in rebalance_signal_dates if d in next_date_map]
    if len(rebalance_signal_dates) == 0:
        raise ValueError("rebalance_signal_dates 为空，请检查交易日期或 REBALANCE_DAYS。")

    rebalance_signal_date_sql = _sql_quote(rebalance_signal_dates)
    rebalance_schedule_df = pd.DataFrame({
        "rebalance_signal_date": rebalance_signal_dates,
        "effective_start_date": [next_date_map[d] for d in rebalance_signal_dates],
    })

    # 仅在调仓信号日计算入选股票，减少 DAI 返回数据量。
    signal_sql = f"""
    WITH factor_base AS (
        SELECT
            date,
            instrument,
            cs_level1_name,
            total_market_cap,
            c_pct_rank(total_market_cap, ascending := true) AS mcap_pct,
            ({num_expr}) / NULLIF(({den_expr}), 0) AS exp_wgt_return_6m
        FROM cn_stock_prefactors
        WHERE date >= '{calc_start_date}'
          AND date <= '{END_DATE}'
          AND list_sector IN (1, 2, 3)
          AND is_risk_warning = 0
          AND suspended = 0
          AND list_days >= {MIN_LIST_DAYS}
          AND close IS NOT NULL
          AND turn IS NOT NULL
          AND total_market_cap IS NOT NULL
          AND cs_level1_name IS NOT NULL
    ),
    with_mcap_group AS (
        SELECT
            *,
            CASE
                WHEN mcap_pct IS NULL THEN NULL
                WHEN mcap_pct >= 1.0 THEN {MCAP_GROUP_COUNT}
                ELSE CAST(FLOOR(mcap_pct * {MCAP_GROUP_COUNT}) + 1 AS INTEGER)
            END AS mcap_group
        FROM factor_base
    ),
    universe AS (
        SELECT
            date,
            instrument,
            total_market_cap,
            mcap_group,
            exp_wgt_return_6m
        FROM with_mcap_group
        WHERE date IN ({rebalance_signal_date_sql})
          AND mcap_group IN ({selected_mcap_group_sql})
          AND cs_level1_name NOT IN ({industry_sql})
          AND exp_wgt_return_6m IS NOT NULL
    ),
    ranked AS (
        SELECT
            date,
            instrument,
            mcap_group,
            exp_wgt_return_6m,
            COUNT(*) OVER (
                PARTITION BY date, mcap_group
            ) AS group_stock_count,
            ROW_NUMBER() OVER (
                PARTITION BY date, mcap_group
                ORDER BY exp_wgt_return_6m ASC, total_market_cap ASC, instrument ASC
            ) AS factor_rank_in_group
        FROM universe
    ),
    selected AS (
        SELECT
            *,
            CAST(CEIL(group_stock_count * {GROUP_SELECT_PCT:.12g}) AS INTEGER) AS group_select_num
        FROM ranked
    )
    SELECT
        date AS rebalance_signal_date,
        instrument,
        mcap_group,
        exp_wgt_return_6m,
        factor_rank_in_group
    FROM selected
    WHERE factor_rank_in_group <= group_select_num
    ORDER BY rebalance_signal_date ASC, mcap_group ASC, factor_rank_in_group ASC, instrument ASC
    """

    target_rebalance_df = dai.query(signal_sql).df()
    target_rebalance_df["rebalance_signal_date"] = pd.to_datetime(
        target_rebalance_df["rebalance_signal_date"]
    ).dt.strftime("%Y-%m-%d")

    if target_rebalance_df.empty:
        raise ValueError(
            "target_rebalance_df 为空，请检查日期范围、市值组别、行业过滤或因子字段。"
        )

    target_rebalance_df = target_rebalance_df.sort_values(
        ["rebalance_signal_date", "mcap_group", "factor_rank_in_group", "instrument"]
    ).reset_index(drop=True)

    target_rebalance_df["instrument"] = target_rebalance_df["instrument"].astype("category")
    target_rebalance_df["mcap_group"] = target_rebalance_df["mcap_group"].astype(np.int16)
    target_rebalance_df["factor_rank_in_group"] = target_rebalance_df["factor_rank_in_group"].astype(np.int32)
    target_rebalance_df["exp_wgt_return_6m"] = target_rebalance_df["exp_wgt_return_6m"].astype(np.float32)

    trade_dates = daily_trend_trade["date"].tolist()
    effective_starts = rebalance_schedule_df["effective_start_date"].tolist()
    effective_start_arr = np.array(effective_starts, dtype=object)

    trade_schedule_df = pd.DataFrame({"date": trade_dates})
    active_idx = np.searchsorted(
        effective_start_arr, trade_schedule_df["date"].values, side="right"
    ) - 1
    trade_schedule_df = trade_schedule_df.loc[active_idx >= 0].copy()
    trade_schedule_df["rebalance_signal_date"] = np.array(
        rebalance_signal_dates, dtype=object
    )[active_idx[active_idx >= 0]]

    daily_target_df = trade_schedule_df.merge(
        target_rebalance_df,
        on="rebalance_signal_date",
        how="inner",
    )

    daily_target_df = daily_target_df.merge(
        daily_trend_trade[["date", "trend_signal_date", "risk_on", "trend_total_weight"]],
        on="date",
        how="left",
    )

    if daily_target_df.empty:
        raise ValueError("daily_target_df 为空，请检查每日目标构造逻辑。")
    if daily_target_df[["risk_on", "trend_total_weight"]].isna().any().any():
        raise ValueError("daily_target_df 中存在缺失的趋势信号，请检查指数趋势数据。")

    daily_target_df["stock_count"] = daily_target_df.groupby(
        "date", observed=True
    )["instrument"].transform("count")
    daily_target_df["weight"] = daily_target_df["trend_total_weight"] / daily_target_df["stock_count"]

    # 只传入 BigTrader 运行必需字段，减少提交模拟时内存占用。
    signal_df = daily_target_df[
        [
            "date",
            "instrument",
            "weight",
            "rebalance_signal_date",
            "trend_signal_date",
            "risk_on",
            "trend_total_weight",
        ]
    ].copy()

    signal_df["date"] = signal_df["date"].astype(str)
    signal_df["instrument"] = signal_df["instrument"].astype(str)
    signal_df["rebalance_signal_date"] = signal_df["rebalance_signal_date"].astype(str)
    signal_df["trend_signal_date"] = signal_df["trend_signal_date"].astype(str)
    signal_df["weight"] = signal_df["weight"].astype(np.float32)
    signal_df["risk_on"] = signal_df["risk_on"].astype(np.int8)
    signal_df["trend_total_weight"] = signal_df["trend_total_weight"].astype(np.float32)

    signal_df = signal_df.sort_values(["date", "instrument"]).reset_index(drop=True)
    if signal_df.empty:
        raise ValueError("signal_df 为空。")

    instruments = sorted(signal_df["instrument"].unique().tolist())
    if len(instruments) == 0:
        raise ValueError("instruments 为空：没有可回测标的，请检查信号生成逻辑。")

    # 实际可运行结束日：END_DATE 可能是非交易日、未来日，或当天数据尚未落库。
    # 使用 signal_df 中真实存在的最后交易日，保证提交模拟每日运行时不会卡在无行情日期。
    effective_end_date = str(signal_df["date"].max())

    return signal_df, instruments, effective_end_date


# =====================================================
# 4. BigTrader 回调函数
# =====================================================

def _current_bar_values(data: bigtrader.IBarData, instrument: str):
    try:
        row = data.current(instrument, ["open", "upper_limit", "lower_limit", "volume"])
    except Exception:
        return None

    try:
        open_price = _safe_float(row["open"])
        upper_limit = _safe_float(row["upper_limit"])
        lower_limit = _safe_float(row["lower_limit"])
        volume = _safe_float(row["volume"], default=0.0)
    except Exception:
        return None

    if pd.isna(open_price) or pd.isna(upper_limit) or pd.isna(lower_limit):
        return None

    return {
        "open": open_price,
        "upper_limit": upper_limit,
        "lower_limit": lower_limit,
        "volume": volume,
    }


def _can_buy_at_current_open(data: bigtrader.IBarData, instrument: str) -> bool:
    bar = _current_bar_values(data, instrument)
    if bar is None or bar["volume"] <= 0:
        return False
    return bar["open"] < bar["upper_limit"] - EPS


def _can_sell_at_current_open(data: bigtrader.IBarData, instrument: str) -> bool:
    bar = _current_bar_values(data, instrument)
    if bar is None or bar["volume"] <= 0:
        return False
    return bar["open"] > bar["lower_limit"] + EPS


def _build_target_maps(signal_data: pd.DataFrame):
    target_by_date = {}
    meta_by_date = {}
    meta_cols = ["rebalance_signal_date", "trend_signal_date", "risk_on", "trend_total_weight"]

    for date, group in signal_data.groupby("date", sort=False):
        target_by_date[date] = dict(
            zip(group["instrument"].values, group["weight"].astype(float).values)
        )
        first = group.iloc[0]
        meta_by_date[date] = {col: first[col] for col in meta_cols}

    return target_by_date, meta_by_date


def _set_current_bar_matching(context: bigtrader.IContext):
    if not USE_CURRENT_BAR_MATCHING:
        return

    try:
        vmatch_enum = getattr(bigtrader, "VMatchAt", None)
        if vmatch_enum is None:
            from bigtrader.constant import VMatchAt
            vmatch_enum = VMatchAt

        if hasattr(vmatch_enum, "CURRENT_BAR"):
            context.set_vmatch_at(vmatch_enum.CURRENT_BAR)
        elif hasattr(vmatch_enum, "CURRENT"):
            context.set_vmatch_at(vmatch_enum.CURRENT)
        else:
            raise AttributeError("BigTrader VMatchAt 中未找到 CURRENT_BAR 或 CURRENT。")
    except Exception as e:
        print("警告：当前 BigTrader 环境未能设置当前 Bar 撮合模式。")
        print("原因：", repr(e))
        print("本策略未使用 m_lead 未来字段；若不能当前 Bar 撮合，开盘涨跌停约束可能无法与成交时点完全对齐。")


def _load_desired_weight_map(context: bigtrader.IContext):
    try:
        stored = context.user_store.get("desired_weight_map", {})
        if isinstance(stored, dict):
            return dict(stored)
    except Exception:
        pass
    return {}


def _save_desired_weight_map(context: bigtrader.IContext):
    try:
        context.user_store["desired_weight_map"] = context.desired_weight_map
    except Exception:
        pass


def _order_succeeded(rv) -> bool:
    # BigTrader 部分环境成功返回 0，部分环境返回 None；其他返回值按失败处理。
    return rv is None or rv == 0


def initialize(context: bigtrader.IContext):
    signal_data = context.data.copy()
    signal_data["date"] = signal_data["date"].astype(str)
    signal_data["instrument"] = signal_data["instrument"].astype(str)

    context.target_by_date, context.meta_by_date = _build_target_maps(signal_data)
    context.desired_weight_map = _load_desired_weight_map(context)

    context.set_commission(
        bigtrader.PerOrder(
            buy_cost=BUY_COST,
            sell_cost=SELL_COST,
            min_cost=MIN_COST,
        )
    )
    _set_current_bar_matching(context)


def handle_data(context: bigtrader.IContext, data: bigtrader.IBarData):
    today = data.current_dt.strftime("%Y-%m-%d")
    target_map = context.target_by_date.get(today)
    if target_map is None:
        return

    if VERBOSE:
        meta = context.meta_by_date.get(today, {})
        print(
            f"{today} 执行：rebalance_signal_date={meta.get('rebalance_signal_date')}, "
            f"trend_signal_date={meta.get('trend_signal_date')}, "
            f"risk_on={meta.get('risk_on')}, "
            f"target_total_weight={float(meta.get('trend_total_weight', 0.0)):.2%}, "
            f"target_count={len(target_map)}"
        )

    current_positions = context.get_positions()
    holding_instruments = set(current_positions.keys())
    target_instruments = set(target_map.keys())

    # 不在目标池的持仓每日尝试卖出；开盘跌停、停牌或无有效行情则跳过。
    for instrument in sorted(holding_instruments - target_instruments):
        if not _can_sell_at_current_open(data, instrument):
            if VERBOSE:
                print(f"{today} 跳过卖出 {instrument}：当前开盘跌停、停牌或无有效行情")
            continue

        rv = context.order_target_percent(instrument, 0)
        if _order_succeeded(rv):
            context.desired_weight_map[instrument] = 0.0
        elif VERBOSE:
            print(f"{today} 卖出 {instrument} 下单失败：rv={rv}")

    # 目标股票：加仓/新开仓要求开盘未涨停；减仓要求开盘未跌停。
    for instrument, target_weight in target_map.items():
        target_weight = float(target_weight)
        prev_weight = float(context.desired_weight_map.get(instrument, 0.0))
        diff = target_weight - prev_weight

        if abs(diff) < 1e-8:
            continue

        if diff > 0:
            if not _can_buy_at_current_open(data, instrument):
                if VERBOSE:
                    print(f"{today} 跳过买入/加仓 {instrument}：当前开盘涨停、停牌或无有效行情")
                continue
        else:
            if not _can_sell_at_current_open(data, instrument):
                if VERBOSE:
                    print(f"{today} 跳过减仓 {instrument}：当前开盘跌停、停牌或无有效行情")
                continue

        rv = context.order_target_percent(instrument, target_weight)
        if _order_succeeded(rv):
            context.desired_weight_map[instrument] = target_weight
        elif VERBOSE:
            print(f"{today} 调整 {instrument} 到 {target_weight:.4%} 下单失败：rv={rv}")

    _save_desired_weight_map(context)


# =====================================================
# 5. 运行 BigTrader：运行成功后可在 AIStudio 中提交模拟
# =====================================================

signal_df, all_target_instruments, EFFECTIVE_END_DATE = build_signal_data()

print(f"动态查询结束日 END_DATE：{END_DATE}")
print(f"实际运行结束日 EFFECTIVE_END_DATE：{EFFECTIVE_END_DATE}")
print(f"传入 BigTrader 的交易日数量：{signal_df['date'].nunique()}，股票数量：{len(all_target_instruments)}")

performance = bigtrader.run(
    market=bigtrader.Market.CN_STOCK,
    frequency=bigtrader.Frequency.DAILY,
    start_date=START_DATE,
    end_date=EFFECTIVE_END_DATE,
    capital_base=CAPITAL_BASE,
    instruments=all_target_instruments,
    data=signal_df,
    initialize=initialize,
    handle_data=handle_data,
    benchmark=BENCHMARK,
    order_price_field_buy="open",
    order_price_field_sell="open",
    volume_limit=0.025,
)


动态查询结束日 END_DATE：2026-09-14
实际运行结束日 EFFECTIVE_END_DATE：2026-09-14
传入 BigTrader 的交易日数量：896，股票数量：259
[2026-09-14 23:19:29] [info     ] bigtrader init ..
[2026-09-14 23:19:29] [info     ] bigtrader.run start: market=cn_stock, frequency=1d, mode=backtest, account_type=STOCK, start_date=2023-01-01, end_date=2026-09-14
[2026-09-14 23:19:30] [info     ] bigtrader<backtest> init ..
[2026-09-14 23:19:30] [info     ] prepare data ..
[2026-09-14 23:19:30] [info     ] bar1d_df: (236130, 16)
[2026-09-14 23:19:30] [info     ] bigtrader use dividend data: (477, 8)
[2026-09-14 23:19:31] [info     ] bigtrader run ..
[2026-09-14 23:19:31] [warning  ] too many logs, only show the last 3000 rows


[2026-09-14 23:19:32] [info     ] bigtrader run done.
